# 🎵 MIDI 데이터 처리 기초

**목표:**
- MIDI 파일 읽기/쓰기
- 노트 정보 추출
- Piano Roll 시각화
- 간단한 멜로디 생성

## 1. 환경 설정

In [ ]:
# 라이브러리 설치
!pip install pretty_midi music21 matplotlib numpy -q

import pretty_midi
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

print("✅ 설치 완료!")

## 2. 샘플 MIDI 다운로드

In [ ]:
# 공개 MIDI 파일 다운로드 (또는 직접 업로드)
!wget -q https://bitmidi.com/uploads/95041.mid -O sample.mid

print("✅ MIDI 파일 다운로드 완료!")

## 3. MIDI 파일 읽기

In [ ]:
# MIDI 파일 로드
midi = pretty_midi.PrettyMIDI('sample.mid')

print(f"곡 길이: {midi.get_end_time():.2f}초")
print(f"악기 수: {len(midi.instruments)}")
print(f"템포 변화: {len(midi.get_tempo_changes()[0])}개")

# 각 악기 정보
for i, instrument in enumerate(midi.instruments):
    instrument_name = pretty_midi.program_to_instrument_name(instrument.program)
    print(f"\nTrack {i}: {instrument_name}")
    print(f"  노트 개수: {len(instrument.notes)}")
    print(f"  드럼 여부: {instrument.is_drum}")

## 4. 노트 정보 추출

In [ ]:
def extract_notes(midi_file):
    """MIDI 파일에서 노트 정보 추출"""
    midi = pretty_midi.PrettyMIDI(midi_file)
    
    notes = []
    for instrument in midi.instruments:
        if not instrument.is_drum:
            for note in instrument.notes:
                notes.append({
                    'pitch': note.pitch,
                    'start': note.start,
                    'end': note.end,
                    'velocity': note.velocity,
                    'duration': note.end - note.start
                })
    
    return notes

# 노트 추출
notes = extract_notes('sample.mid')
print(f"총 노트 수: {len(notes)}")
print(f"\n첫 5개 노트:")
for note in notes[:5]:
    pitch_name = pretty_midi.note_number_to_name(note['pitch'])
    print(f"  {pitch_name} (피치: {note['pitch']}, 시작: {note['start']:.2f}초, 길이: {note['duration']:.2f}초)")

## 5. Piano Roll 시각화

In [ ]:
def plot_piano_roll(notes, max_time=10, title="Piano Roll"):
    """피아노 롤 시각화"""
    # 시간 제한
    notes_subset = [n for n in notes if n['start'] < max_time]
    
    fig, ax = plt.subplots(figsize=(15, 5))
    
    for note in notes_subset:
        ax.barh(
            note['pitch'],
            width=note['duration'],
            left=note['start'],
            height=0.8,
            alpha=0.8,
            color='steelblue'
        )
    
    ax.set_xlabel('Time (seconds)', fontsize=12)
    ax.set_ylabel('MIDI Pitch', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

# 시각화
plot_piano_roll(notes, max_time=10, title="첫 10초 Piano Roll")

## 6. 피치 분포 분석

In [ ]:
# 피치 분포
pitches = [n['pitch'] for n in notes]

plt.figure(figsize=(12, 5))

# 히스토그램
plt.subplot(1, 2, 1)
plt.hist(pitches, bins=50, edgecolor='black')
plt.xlabel('MIDI Pitch')
plt.ylabel('Frequency')
plt.title('Note Distribution')
plt.grid(alpha=0.3)

# 통계
plt.subplot(1, 2, 2)
stats = {
    'Min Pitch': min(pitches),
    'Max Pitch': max(pitches),
    'Mean Pitch': np.mean(pitches),
    'Median Pitch': np.median(pitches)
}
plt.barh(list(stats.keys()), list(stats.values()))
plt.xlabel('Value')
plt.title('Pitch Statistics')

plt.tight_layout()
plt.show()

print(f"음역대: {pretty_midi.note_number_to_name(min(pitches))} ~ {pretty_midi.note_number_to_name(max(pitches))}")

## 7. 간단한 멜로디 생성

In [ ]:
def create_melody(pitches, durations, velocities=None, tempo=120):
    """
    간단한 멜로디 생성
    
    Args:
        pitches: MIDI 피치 리스트
        durations: 각 노트의 길이 (초)
        velocities: 각 노트의 세기 (옵션)
        tempo: BPM
    """
    midi = pretty_midi.PrettyMIDI(initial_tempo=tempo)
    piano = pretty_midi.Instrument(program=0)  # Acoustic Grand Piano
    
    if velocities is None:
        velocities = [100] * len(pitches)
    
    current_time = 0
    for pitch, duration, velocity in zip(pitches, durations, velocities):
        note = pretty_midi.Note(
            velocity=velocity,
            pitch=pitch,
            start=current_time,
            end=current_time + duration
        )
        piano.notes.append(note)
        current_time += duration
    
    midi.instruments.append(piano)
    return midi

# 예제 1: C major scale
c_major_scale = [60, 62, 64, 65, 67, 69, 71, 72]  # C D E F G A B C
durations = [0.5] * 8

midi_scale = create_melody(c_major_scale, durations)
midi_scale.write('c_major_scale.mid')

# 재생
audio = midi_scale.fluidsynth()
display(Audio(audio, rate=44100))

print("✅ C major scale 생성 완료!")

In [ ]:
# 예제 2: "Twinkle Twinkle Little Star"
twinkle = [
    60, 60, 67, 67, 69, 69, 67,  # Twinkle twinkle little star
    65, 65, 64, 64, 62, 62, 60   # How I wonder what you are
]
twinkle_durations = [
    0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 1.0,
    0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 1.0
]

midi_twinkle = create_melody(twinkle, twinkle_durations)
midi_twinkle.write('twinkle.mid')

# 재생
audio = midi_twinkle.fluidsynth()
display(Audio(audio, rate=44100))

print("✅ Twinkle Twinkle Little Star 생성 완료!")

## 8. MIDI → Piano Roll 변환

In [ ]:
def midi_to_piano_roll(midi_file, time_step=0.125):
    """
    MIDI 파일을 Piano Roll로 변환
    
    time_step: 시간 간격 (초), 0.125 = 32분음표
    """
    midi = pretty_midi.PrettyMIDI(midi_file)
    
    # Piano roll 생성
    piano_roll = midi.get_piano_roll(fs=1/time_step)
    
    # 이진화 (0/1)
    piano_roll = (piano_roll > 0).astype(int)
    
    return piano_roll

# 변환
piano_roll = midi_to_piano_roll('sample.mid')
print(f"Piano Roll Shape: {piano_roll.shape}")  # (128, time_steps)

# 시각화
plt.figure(figsize=(15, 5))
plt.imshow(piano_roll[:, :400], aspect='auto', cmap='binary', origin='lower')
plt.xlabel('Time Steps (0.125초 간격)')
plt.ylabel('MIDI Pitch')
plt.title('Piano Roll Representation')
plt.colorbar(label='Active (1) / Inactive (0)')
plt.show()

## 9. 연습 문제

### 문제 1: 자신만의 멜로디 만들기
- 8개 이상의 노트로 멜로디 작곡
- MIDI 파일로 저장
- Piano Roll 시각화

In [ ]:
# 여기에 코드 작성
my_pitches = [60, 62, 64, 65, 67]  # 수정하세요
my_durations = [0.5, 0.5, 0.5, 0.5, 1.0]  # 수정하세요

# TODO: 멜로디 생성 및 저장

### 문제 2: MIDI 파일 분석
- 샘플 MIDI 파일에서 가장 많이 사용된 피치 5개 찾기
- 평균 노트 길이 계산

In [ ]:
# 여기에 코드 작성
from collections import Counter

# TODO: 가장 많이 사용된 피치
pitch_counts = Counter(pitches)
top_5 = pitch_counts.most_common(5)
print("가장 많이 사용된 피치 5개:")
for pitch, count in top_5:
    print(f"  {pretty_midi.note_number_to_name(pitch)}: {count}회")

# TODO: 평균 노트 길이
avg_duration = np.mean([n['duration'] for n in notes])
print(f"\n평균 노트 길이: {avg_duration:.3f}초")

## 10. 정리

**배운 내용:**
- ✅ MIDI 파일 읽기 및 노트 정보 추출
- ✅ Piano Roll 시각화
- ✅ 간단한 멜로디 생성
- ✅ MIDI → Piano Roll 변환

**다음 단계:**
- LSTM으로 멜로디 학습 및 생성
- Transformer 모델 구현

**화이팅!** 🎵